In [15]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from pathlib import Path
import numpy as np
import re

In [16]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\theoheuner\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\theoheuner\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\theoheuner\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\theoheuner\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Reading the data from the file

In [17]:
sentences_file = open(Path.cwd() / "FinancialPhraseBank-v1.0" / "Sentences_50Agree.txt")
raw_sentences = sentences_file.read()

sentences_list_with_sentiments = raw_sentences.split("\n")[:-1]

### Splitting the the raw text into the actual text and labels

In [18]:
sentences_list = []
labels = []
sentiment_dictionary = {"negative": -1, "neutral": 0, "positive": 1}

for sentence in sentences_list_with_sentiments:
    text, sentiment = sentence.rsplit('@', 1)
    sentences_list.append(text)
    labels.append(sentiment_dictionary[sentiment])
labels = np.array(labels)

In [19]:
print(f"Number of sentences with negative annotation: {len(np.where(labels == -1)[0])}")
print(f"Number of sentences with neutral annotation: {len(np.where(labels == 0)[0])}")
print(f"Number of sentences with positive annotation: {len(np.where(labels == 1)[0])}")

Number of sentences with negative annotation: 604
Number of sentences with neutral annotation: 2879
Number of sentences with positive annotation: 1363


### Lowercasing

In [20]:
sentences_list_all_lower = [sentence.lower() for sentence in sentences_list]

### Noise removal

In [21]:
noise_characters_dict = {"%": "", ",":"", "(":"", ")":"", ":":"", "n't":"not", "'s":"", "eur":"", "€":"", "$":"", "``":"", "''":"", "...":"", "  ":" "}
sentences_list_denoised = sentences_list_all_lower
for noise_character, replace_character in noise_characters_dict.items():
    sentences_list_denoised = [sentence.replace(noise_character, replace_character) for sentence in sentences_list_denoised]

### Stop word removal

In [22]:
nltk_stop_words = set(stopwords.words('english'))
not_stop_words = {"above", "after", "against", "before", "below", "between", "down", "no", "not", "nor", "off", "only", "over", "same", "under", "up"}
additional_stop_words = {"bln", "mln", "pct", "percent"}
stop_words = (nltk_stop_words - not_stop_words).union(additional_stop_words)
stop_words

{'a',
 'about',
 'again',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'being',
 'bln',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'mln',
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'now',
 'o',
 'of',
 'on',
 'once',
 'or',
 'other',
 'our',
 'ours',
 'ourselves',
 'out',
 'own',
 'pct',
 'percent',
 're',
 '

In [23]:
sentences_list_without_stop_words = []
for sentence in sentences_list_denoised:
    words_list = word_tokenize(sentence)
    sentence_without_stop_words = " ".join([word for word in words_list if word not in stop_words])
    sentences_list_without_stop_words.append(sentence_without_stop_words)

### Lemmatization

In [24]:
lemmatizer = WordNetLemmatizer()

sentences_list_lemmatized = []
for sentence in sentences_list_without_stop_words:
    words_list = word_tokenize(sentence)
    sentence_lemmatized = " ".join([lemmatizer.lemmatize(word) for word in words_list])
    sentences_list_lemmatized.append(sentence_lemmatized)

### (Optional) Removing numbers

In [25]:
sentences_list_without_numbers = []
for sentence in sentences_list_lemmatized:
    sentence_without_numbers = re.sub(r'\b\d+\s*(?:[\.,]\s*\d+)?\b', '<NUM>', sentence)
    sentences_list_without_numbers.append(sentence_without_numbers)

### Function for whole pre-processing

In [26]:
def preprocess_sentences(sentences_list):
    sentences_list_all_lower = [sentence.lower() for sentence in sentences_list]

    noise_characters_dict = {"%": "", ",":"", "(":"", ")":"", ":":"", "n't":"not", "'s":"", "eur":"", "€":"", "$":"", "``":"", "''":"", "...":"", "  ":" "}
    sentences_list_denoised = sentences_list_all_lower
    for noise_character, replace_character in noise_characters_dict.items():
        sentences_list_denoised = [sentence.replace(noise_character, replace_character) for sentence in sentences_list_denoised]

    nltk_stop_words = set(stopwords.words('english'))
    not_stop_words = {"above", "after", "against", "before", "below", "between", "down", "no", "not", "nor", "off", "only", "over", "same", "under", "up"}
    additional_stop_words = {"bln", "mln", "pct", "percent"}
    stop_words = (nltk_stop_words - not_stop_words).union(additional_stop_words)
    sentences_list_without_stop_words = []
    for sentence in sentences_list_denoised:
        words_list = word_tokenize(sentence)
        sentence_without_stop_words = " ".join([word for word in words_list if word not in stop_words])
        sentences_list_without_stop_words.append(sentence_without_stop_words)
    
    lemmatizer = WordNetLemmatizer()
    sentences_list_lemmatized = []
    for sentence in sentences_list_without_stop_words:
        words_list = word_tokenize(sentence)
        sentence_lemmatized = " ".join([lemmatizer.lemmatize(word) for word in words_list])
        sentences_list_lemmatized.append(sentence_lemmatized)


    sentences_list_without_numbers = []
    for sentence in sentences_list_lemmatized:
        sentence_without_numbers = re.sub(r'\b\d+\s*(?:[\.,]\s*\d+)?\b', '<NUM>', sentence)
        sentences_list_without_numbers.append(sentence_without_numbers)

    return sentences_list_all_lower, sentences_list_denoised, sentences_list_without_stop_words, sentences_list_lemmatized, sentences_list_without_numbers

In [27]:
test_sentence_id = 317
print("Original Sentence:")
print(sentences_list[test_sentence_id])
print("Lowercase Sentence:")
print(sentences_list_all_lower[test_sentence_id])
print("Denoised Sentence:")
print(sentences_list_denoised[test_sentence_id])
print("Sentence Without Stop Words:")
print(sentences_list_without_stop_words[test_sentence_id])
print("Lemmatized Sentence:")
print(sentences_list_lemmatized[test_sentence_id])
print("Sentence Without Numbers:")
print(sentences_list_without_numbers[test_sentence_id])
print(labels[test_sentence_id])

Original Sentence:
`` The transaction strengthens our position ... in design and branded goods , '' said Fiskars president and CEO Heikki Allonen , pointing out that the two groups have relatively few overlapping operations .
Lowercase Sentence:
`` the transaction strengthens our position ... in design and branded goods , '' said fiskars president and ceo heikki allonen , pointing out that the two groups have relatively few overlapping operations .
Denoised Sentence:
 the transaction strengthens our position in design and branded goods  said fiskars president and ceo heikki allonen pointing out that the two groups have relatively few overlapping operations .
Sentence Without Stop Words:
transaction strengthens position design branded goods said fiskars president ceo heikki allonen pointing two groups relatively overlapping operations .
Lemmatized Sentence:
transaction strengthens position design branded good said fiskars president ceo heikki allonen pointing two group relatively overla